In [ ]:
import os
import pandas as pd 
import xarray as xr
import numpy as np
from datetime import datetime

from scipy.stats import ttest_1samp
from scipy.ndimage import gaussian_filter

import cmocean
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.colors as mcolors

import xskillscore as xs

In [ ]:
class SoilMoistureDifferencePlotter:
    def __init__(self, data_dict, landmask_file=None, soilayer_file=None,
                 variable='H2OSOI', dz_var='DZSOI', landmask_var='landfrac',
                 depth_cm=10, mask_land=True, confidence_level=0.05,
                 use_mass_units=True):
        self.data_dict = data_dict
        self.variable = variable
        self.dz_var = dz_var
        self.landmask_var = landmask_var
        self.depth_cm = depth_cm
        self.mask_land = mask_land
        self.significance_threshold = 0
        self.confidence_level = confidence_level
        self.use_mass_units = use_mass_units
        self.water_density = 1000.0

        if landmask_file and os.path.exists(landmask_file):
            landmask = xr.open_dataset(landmask_file)[landmask_var]
            landmask = self.normalize_longitude_and_sort(landmask)
            self.landmask = landmask.where(landmask > 0.1) if mask_land else landmask.where(landmask <= 0.5)

        self.fixed_dz = None
        if soilayer_file and os.path.exists(soilayer_file):
            self.fixed_dz = xr.open_dataset(soilayer_file)[dz_var]

    def normalize_longitude_and_sort(self, ds, lon_name='lon'):
        if lon_name in ds.coords:
            lon = ds[lon_name]
            if lon.max() > 180:
                print("[INFO] Normalizing longitude to [-180, 180].")
                ds[lon_name] = ((lon + 180) % 360) - 180
                ds = ds.sortby(lon_name)
        return ds

    def _wrap_longitude_for_plot(self, da):
        if 'lon' not in da.coords:
            return da
        da = da.sortby('lon')
        lon = da['lon'].values
        dlon = lon[1] - lon[0]
        if lon[-1] - lon[0] < 350:
            new_lon = np.append(lon, lon[-1] + dlon)
            new_data = np.concatenate([da.values, da.isel(lon=0).values[..., np.newaxis]], axis=-1)
            da_wrapped = xr.DataArray(
                data=new_data,
                dims=da.dims,
                coords={**{k: da.coords[k] for k in da.dims if k != 'lon'}, 'lon': new_lon},
                attrs=da.attrs,
                name=da.name if hasattr(da, 'name') else None
            )
            return da_wrapped
        return da

    def _load_dataset(self, info, year, ensemble=None):
        template = info['template']
        path = info['path']
        filename = template % {'year': year, 'ensemble': ensemble} if ensemble else template % {'year': year}
        full_path = os.path.join(path, filename)
        if not os.path.exists(full_path):
            raise FileNotFoundError(f"[ERROR] File not found: {full_path}")
        return xr.open_dataset(full_path)

    def _integrate_top_layers(self, ds):
        sm = ds[self.variable]
        dz = ds[self.dz_var] if self.dz_var in ds else self.fixed_dz
        if dz is None:
            raise ValueError("No valid DZSOI found in dataset or external source.")
        if "time" in dz.dims:
            dz = dz.isel(time=0)
        dz_cumsum = dz.cumsum(dim='levgrnd')
        top_mask = dz_cumsum <= (self.depth_cm / 100.0)
        sm_top = sm.where(top_mask)
        dz_masked = dz.where(top_mask)
        if self.use_mass_units:
            sm_mass = sm_top * dz_masked * self.water_density
            integrated = sm_mass.sum(dim='levgrnd')
            integrated.attrs['units'] = 'kg/m2'
        else:
            integrated = (sm_top * dz_masked).sum(dim='levgrnd') / dz_masked.sum(dim='levgrnd')
            integrated.attrs['units'] = 'm3/m3'
            
        if self.landmask is not None:
            integrated = integrated.where(self.landmask > 0.1)
        return integrated

    def _load_model_ensemble_snapshot(self, model_info, target_date):
        nens = model_info['nens']
        ens_data = []
        for n in range(1, nens + 1):
            ens = f'EN{n:02d}'
            ds = self._load_dataset(model_info, target_date.year, ensemble=ens)
            ds = self.normalize_longitude_and_sort(ds)
            ds_day = ds.sel(time=target_date)
            sm_top = self._integrate_top_layers(ds_day)
            ens_data.append(sm_top.squeeze())
        return xr.concat(ens_data, dim='ensemble')

    def _load_obs_snapshot(self, obs_info, target_date):
        ds = self._load_dataset(obs_info, target_date.year)
        ds = self.normalize_longitude_and_sort(ds)
        obs = ds[self.variable].sel(time=target_date)
        if self.use_mass_units:
            obs = obs * 1000.0 * self.depth_cm / 100.0
        if self.landmask is not None:
            obs = obs.where(self.landmask > 0.1)
        return obs

    def _compute_significance_mask(self, ensemble_data, obs_mean, alpha=0.05, threshold=None):
        if threshold is None:
            threshold = self.significance_threshold
        diff = ensemble_data - obs_mean
        tstat, pvals = ttest_1samp(diff.transpose("ensemble", "lat", "lon").values,
                                   popmean=0.0, axis=0, nan_policy='omit')
        sig_mask = xr.DataArray(pvals < alpha, coords=obs_mean.coords, dims=obs_mean.dims)
        model_mean = ensemble_data.mean(dim='ensemble')
        valid_mask = (abs(model_mean) > threshold) & (abs(obs_mean) > threshold)
        return sig_mask.where(valid_mask)

    def plot_bias_and_stddev_multi_panel(self, model_keys, target_date='2012-01-01',
                                         obs_key='ESA_CCI',
                                         bias_levels=None, spread_levels=None,
                                         cmap_bias='RdBu_r', cmap_std='viridis',
                                         figsize=(12, 5), savepath=None,
                                         wrap_longitude=True, smooth_sigma=1.0):
        target_date = pd.to_datetime(target_date)
        ncols = len(model_keys)

        bias_levels = bias_levels or [-0.5, -0.3, -0.1, -0.05, -0.01, 0.01, 0.05, 0.1, 0.3, 0.5]
        spread_levels = spread_levels or [0.005, 0.01, 0.02, 0.04, 0.06, 0.08, 0.1]

        bias_cmap = plt.get_cmap(cmap_bias, len(bias_levels) - 1)
        bias_norm = mcolors.BoundaryNorm(bias_levels, ncolors=bias_cmap.N)
        spread_cmap = plt.get_cmap(cmap_std, len(spread_levels) - 1)
        spread_norm = mcolors.BoundaryNorm(spread_levels, ncolors=spread_cmap.N)

        fig, axes = plt.subplots(nrows=2, ncols=ncols, figsize=figsize,
                                 subplot_kw={'projection': ccrs.Robinson()},
                                 constrained_layout=False)

        for j, model_key in enumerate(model_keys):
            model_info = self.data_dict[model_key]
            obs_info = self.data_dict[obs_key]

            ens_stack = self._load_model_ensemble_snapshot(model_info, target_date)
            obs_field = self._load_obs_snapshot(obs_info, target_date)
            if ens_stack.shape[1:] != obs_field.shape:
                ens_stack = ens_stack.interp_like(obs_field)

            model_mean = ens_stack.mean(dim='ensemble')
            model_std = ens_stack.std(dim='ensemble')
            bias = model_mean - obs_field

            if wrap_longitude:
                bias = self._wrap_longitude_for_plot(bias)
                model_std = self._wrap_longitude_for_plot(model_std)

            # Apply smoothing
            if smooth_sigma:
                bias.values = gaussian_filter(bias.values, sigma=smooth_sigma)
                model_std.values = gaussian_filter(model_std.values, sigma=smooth_sigma)

            # PCC and RMSE
            valid = np.isfinite(model_mean.values.flatten()) & np.isfinite(obs_field.values.flatten())
            pcc = np.corrcoef(model_mean.values.flatten()[valid], obs_field.values.flatten()[valid])[0, 1]
            rmse_val = xs.rmse(model_mean, obs_field, dim=['lat', 'lon'], skipna=True)            
            # Ensure lat-lon match and flatten both to 1D
            model_flat = model_mean.stack(space=('lat', 'lon'))
            obs_flat = obs_field.stack(space=('lat', 'lon'))
            # Optionally drop NaNs where either is missing
            valid = model_flat.notnull() & obs_flat.notnull()
            # Compute PCC
            pcc = xs.pearson_r(model_flat.where(valid), obs_flat.where(valid), dim='space', skipna=True)
            print(f"PCC over spatial domain: {pcc.values:.3f}")

            # Plot Bias
            ax0 = axes[0, j]
            ax0.contourf(bias['lon'], bias['lat'], bias,
                         levels=bias_levels, cmap=bias_cmap, norm=bias_norm,
                         transform=ccrs.PlateCarree())
            ax0.set_title(f"{model_key} - {obs_key} (Bias)", fontsize=10)
            ax0.coastlines()
            ax0.set_global()
            ax0.tick_params(labelsize=8)
            ax0.text(0.98, 0.02, f"PCC: {pcc:.2f}\nRMSE: {rmse_val:.2f}",
                     transform=ax0.transAxes, fontsize=8, va='bottom', ha='right',
                     bbox=dict(facecolor='white', edgecolor='black', boxstyle='round,pad=0.3'))

            sig_mask = self._compute_significance_mask(ens_stack, obs_field, alpha=self.confidence_level)
            if wrap_longitude:
                sig_mask = self._wrap_longitude_for_plot(sig_mask)
            ax0.contourf(sig_mask['lon'], sig_mask['lat'], sig_mask,
                         levels=[0.5, 1], colors='none', hatches=['...', None],
                         transform=ccrs.PlateCarree())

            # Plot Spread
            ax1 = axes[1, j]
            ax1.contourf(model_std['lon'], model_std['lat'], model_std,
                         levels=spread_levels, cmap=spread_cmap, norm=spread_norm,
                         transform=ccrs.PlateCarree())
            ax1.set_title(f"{model_key} (Spread)", fontsize=10)
            ax1.coastlines()
            ax1.set_global()
            ax1.tick_params(labelsize=8)

        # Colorbars
        cbar_ax1 = fig.add_axes([0.90, 0.60, 0.012, 0.22])
        fig.colorbar(mpl.cm.ScalarMappable(norm=bias_norm, cmap=bias_cmap),
                     cax=cbar_ax1, ticks=bias_levels,
                     label="Bias (kg m$^{-2}$)" if self.use_mass_units else "Bias (m$^3$ m$^{-3}$)")

        cbar_ax2 = fig.add_axes([0.90, 0.22, 0.012, 0.22])
        fig.colorbar(mpl.cm.ScalarMappable(norm=spread_norm, cmap=spread_cmap),
                     cax=cbar_ax2, ticks=spread_levels,
                     label="Spread (kg m$^{-2}$)" if self.use_mass_units else "Spread (m$^3$ m$^{-3}$)")

        plt.subplots_adjust(hspace=0.15, wspace=0.05)
        plt.show()

        if savepath:
            fig.savefig(savepath, dpi=600, bbox_inches='tight')
            print(f"[SAVED] Figure saved to: {savepath}")


In [ ]:
if __name__ == "__main__":
    top_path = "/pscratch/sd/z/zhan391/e3sm_dart"
    out_path = os.path.join(top_path, "diag_dart")
    fig_path = "/global/homes/z/zhan391/analysis/diagnostic/figures"
    os.makedirs(fig_path, exist_ok=True)

    landmask_file = os.path.join(out_path, "landmask_1x1.nc")
    soilayer_file = os.path.join(out_path, "dzsoi_elm.nc")

    data_dict = {
        'CPC_SOM': {
            'path': f'{top_path}/Observations/CPC_SOM/monthly',
            'template': 'SOILWATER_10CM.daily.%(year)s.nc',
            'frequency': 'monthly',
            'nens': 1
        },
        'ESA_CCI': {
            'path': f'{top_path}/Observations/ESA_CCI/daily',
            'template': 'H2OSOI.daily.%(year)s.nc',
            'frequency': 'daily',
            'nens': 1
        },
        'CTRLEN10': {
            'path': f'{top_path}/CTRLEN10_15day_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy/archive/post/lnd/180x360_aave/ts/daily',
            'template': 'H2OSOI.%(ensemble)s.%(year)s.nc',
            'frequency': 'daily',
            'nens': 10
        },
        'CAPTEN10': {
            'path': f'{top_path}/CTRLEN10_15day_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy/archive/post/lnd/180x360_aave/ts/daily',
            'template': 'H2OSOI.%(ensemble)s.%(year)s.nc',
            'frequency': 'daily',
            'nens': 10
        },
        'DARTEN20': {
            'path': f'{top_path}/DARTEN20_15day_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy/archive/post/lnd/180x360_aave/ts/daily',
            'template': 'H2OSOI.%(ensemble)s.%(year)s.nc',
            'frequency': 'daily',
            'nens': 20
        },
        'DARTEN40': {
            'path': f'{top_path}/DARTEN40_15day_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy/archive/post/lnd/180x360_aave/ts/daily',
            'template': 'H2OSOI.%(ensemble)s.%(year)s.nc',
            'frequency': 'daily',
            'nens': 40
        }
    }

    model_keys = ['CAPTEN10', 'DARTEN20', 'DARTEN40'] #'CTRLEN10', 
    target_date = '2012-01-01'
    obs_key = 'ESA_CCI'
    bias_levels=[-5.0, -2.0, -1.0, -0.5, -0.1, -0.01, 0, 0.01, 0.1, 0.5, 1, 2, 5] 
    spread_levels=[0.01, 0.02, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5]

    bias_levels = [a/100 for a in bias_levels ]
    spread_levels = [b/100 for b in spread_levels ]
    
    # Combined bias + std dev plot
    combined_savefile = os.path.join(
        fig_path, f"sm_bias_stddev_combined_{target_date.replace('-', '')}.pdf"
    )
    
    plotter = SoilMoistureDifferencePlotter(
        data_dict=data_dict,
        landmask_file=landmask_file,
        soilayer_file=soilayer_file,
        depth_cm=5,
        mask_land=False,
        confidence_level = 0.05,
        use_mass_units=False 
    )
    
    plotter.plot_bias_and_stddev_multi_panel(
        model_keys=model_keys,
        target_date=target_date,
        obs_key=obs_key,
        bias_levels=bias_levels,
        spread_levels=spread_levels,
        figsize=(12,6),
        cmap_bias=cmocean.cm.curl, 
        cmap_std=cmocean.cm.deep,
        savepath=combined_savefile
    )